# Trivium Stream Cipher
**Implementation of the Trivium cipher for key stream generation and OTP encryption/decryption.**

---
### Overview
- **Key size:** Up to 80 bits
- **IV size:** Up to 80 bits (randomly generated nonce)
- **Internal state:** 288 bits
- **Warm-up rounds:** 1152
- **Encryption:** One-Time Pad (XOR with keystream)

## Step 1: Imports and Helper Functions

In [41]:
import os

def hex_to_bits(hex_str, num_bits):
    """
    Convert a hex string to a list of bits (LSB-first, padded to num_bits).
    Trivium loads key/IV in LSB-first order.
    """
    value = int(hex_str, 16)
    bits = []
    for _ in range(num_bits):
        bits.append(value & 1)
        value >>= 1
    return bits  # length == num_bits, LSB first


def bits_to_hex(bits):
    """
    Convert a list of bits (MSB-first) to a hex string.
    Used for displaying the keystream and ciphertext.
    """
    # Pack bits into bytes (MSB first within each byte)
    result = []
    for i in range(0, len(bits), 8):
        byte_bits = bits[i:i+8]
        byte_val = 0
        for j, b in enumerate(byte_bits):
            byte_val |= (b << (7 - j))
        result.append(byte_val)
    return bytes(result).hex().upper()


def generate_nonce():
    """Generate a random 80-bit (10-byte) nonce."""
    return os.urandom(10).hex().upper()


def text_to_bits(text):
    """Convert ASCII text to a list of bits (MSB-first per byte)."""
    bits = []
    for char in text:
        byte_val = ord(char)
        for j in range(7, -1, -1):   # MSB first
            bits.append((byte_val >> j) & 1)
    return bits


def bits_to_text(bits):
    """Convert a list of bits (MSB-first per byte) back to ASCII text."""
    chars = []
    for i in range(0, len(bits), 8):
        byte_bits = bits[i:i+8]
        byte_val = 0
        for j, b in enumerate(byte_bits):
            byte_val |= (b << (7 - j))
        chars.append(chr(byte_val))
    return ''.join(chars)


def hex_keystream_to_bits(hex_str, num_bits):
    """
    Convert a hex keystream string back into a flat list of bits (MSB-first per byte).
    Used during decryption.
    """
    bits = []
    for i in range(0, len(hex_str), 2):
        byte_val = int(hex_str[i:i+2], 16)
        for j in range(7, -1, -1):
            bits.append((byte_val >> j) & 1)
    return bits[:num_bits]


print("Helper functions loaded.")

Helper functions loaded.


## Step 2: Trivium Core Implementation

In [42]:
# ─────────────────────────────────────────────
# TRIVIUM CIPHER
# ─────────────────────────────────────────────

def trivium_init(key_hex, iv_hex):
    """
    Build and warm up the 288-bit Trivium internal state.

    Layout (1-based, spec notation):
        Register A : S[001] – S[093]   (bits 0-92  in Python)
        Register B : S[094] – S[177]   (bits 93-176 in Python)
        Register C : S[178] – S[288]   (bits 177-287 in Python)

    Initialisation:
        Key  → S[001]–S[080]  (Python indices 0-79)
        IV   → S[094]–S[173]  (Python indices 93-172)
        111  → S[286]–S[288]  (Python indices 285-287)
        All other bits = 0.
    """
    # build state
    state = [0] * 288

    # Load key (LSB-first, up to 80 bits) into S[1..80] → indices 0-79
    key_bits = hex_to_bits(key_hex, 80)
    for i in range(80):
        state[i] = key_bits[i]

    # Load IV (LSB-first, up to 80 bits) into S[94..173] → indices 93-172
    iv_bits = hex_to_bits(iv_hex, 80)
    for i in range(80):
        state[93 + i] = iv_bits[i]

    # Set S[286..288] = 1  →  indices 285-287
    state[285] = 1
    state[286] = 1
    state[287] = 1

    # 1152 warm-up rounds (no output collected)
    for _ in range(1152):
        state = _trivium_round(state)

    return state


def _trivium_round(s):
    """
    Execute one Trivium round on state s (0-based).
    Returns (output_bit, new_state).

    Spec formula (1-based) translated to 0-based:
        t1 = S[066] ^ S[093]  →  s[65]  ^ s[92]
        t2 = S[162] ^ S[177]  →  s[161] ^ s[176]
        t3 = S[243] ^ S[288]  →  s[242] ^ s[287]

        output_bit = t1 ^ t2 ^ t3

        t1 = t1 ^ (S[091] & S[092]) ^ S[171]  →  ... ^ (s[90]&s[91]) ^ s[170]
        t2 = t2 ^ (S[175] & S[176]) ^ S[264]  →  ... ^ (s[174]&s[175]) ^ s[263]
        t3 = t3 ^ (S[286] & S[287]) ^ S[069]  →  ... ^ (s[285]&s[286]) ^ s[68]

        Right-shift state by 1:
            S[001] ← t3   →  new_s[0]   = t3
            S[094] ← t1   →  new_s[93]  = t1
            S[178] ← t2   →  new_s[177] = t2
    """
    t1 = s[65]  ^ s[92]
    t2 = s[161] ^ s[176]
    t3 = s[242] ^ s[287]

    output_bit = t1 ^ t2 ^ t3

    t1 = t1 ^ (s[90] & s[91]) ^ s[170]
    t2 = t2 ^ (s[174] & s[175]) ^ s[263]
    t3 = t3 ^ (s[285] & s[286]) ^ s[68]

    # Right-shift: s[i] ← s[i-1] for i = 287 downto 1, then inject new bits
    new_s = [0] + s[:-1]   # shift everything right by one position

    new_s[0]   = t3   # S_001 ← t3
    new_s[93]  = t1   # S_094 ← t1
    new_s[177] = t2   # S_178 ← t2

    return new_s, output_bit


# Patch: make _trivium_round return (new_state, bit) and update trivium_init
def _trivium_round(s):
    t1 = s[65]  ^ s[92]
    t2 = s[161] ^ s[176]
    t3 = s[242] ^ s[287]

    output_bit = t1 ^ t2 ^ t3

    t1 = t1 ^ (s[90] & s[91]) ^ s[170]
    t2 = t2 ^ (s[174] & s[175]) ^ s[263]
    t3 = t3 ^ (s[285] & s[286]) ^ s[68]

    new_s = [0] + s[:-1]
    new_s[0]   = t3
    new_s[93]  = t1
    new_s[177] = t2

    return new_s, output_bit


def trivium_init(key_hex, iv_hex):
    state = [0] * 288

    key_bits = hex_to_bits(key_hex, 80)
    for i in range(80):
        state[i] = key_bits[i]

    iv_bits = hex_to_bits(iv_hex, 80)
    for i in range(80):
        state[93 + i] = iv_bits[i]

    state[285] = 1
    state[286] = 1
    state[287] = 1

    for _ in range(1152):
        state, _ = _trivium_round(state)   # discard output during warm-up

    return state    


def trivium_generate_keystream(state, num_bits):
    """
    Generate `num_bits` keystream bits starting from the warmed-up state.
    Returns the keystream as a list of bits.
    """
    keystream = []
    for _ in range(num_bits):
        state, bit = _trivium_round(state)
        keystream.append(bit)
    return keystream


print("Trivium core loaded.")

Trivium core loaded.


## Step 3: Encryption (Part 1)

In [43]:
# ─────────────────── #
# PART 1 — ENCRYPTION #
# ─────────────────── #

DEFAULT_KEY_HEX = 'FF' * 10   # 80 bits of all 1s

print("=" * 55)
print("         TRIVIUM ENCRYPTION (Part 1)")
print("=" * 55)

# Plaintext
plaintext = input("Enter plaintext: ")

# Key 
use_default = input("Use default key (all 1-bits)? [y/n]: ").strip().lower()

if use_default == 'y':
    key_hex = DEFAULT_KEY_HEX
    print(f"Using default key: {key_hex}")
else:
    raw_key = input("Enter key in HEX (up to 20 hex chars = 80 bits): ").strip().upper()
    # Pad to 20 hex characters (80 bits) with leading zeros on the RIGHT
    # (spec: shorter key → fill remaining bits with 0)
    # Each hex char = 4 bits, so 80 bits = 20 hex chars
    raw_key = raw_key[:20]                    # truncate if too long
    key_hex = raw_key.ljust(20, '0')          # pad with 0s on the right
    print(f"Key (padded to 80-bit): {key_hex}")

#  Nonce / IV 
nonce_hex = generate_nonce()
print(f"Generated Nonce/IV:     {nonce_hex}")

#  Trivium keystream generation 
plaintext_bits = text_to_bits(plaintext)
num_bits = len(plaintext_bits)                # 8 * len(plaintext)

print(f"\nPlaintext length : {len(plaintext)} chars → {num_bits} bits")
print("Initialising Trivium (1152 warm-up rounds)...")

state = trivium_init(key_hex, nonce_hex)
keystream_bits = trivium_generate_keystream(state, num_bits)

keystream_hex = bits_to_hex(keystream_bits)
print(f"\nOutput Key Stream : {keystream_hex}")

# OTP Encryption: ciphertext = plaintext XOR keystream 
cipher_bits = [p ^ k for p, k in zip(plaintext_bits, keystream_bits)]
ciphertext_hex = bits_to_hex(cipher_bits)

print(f"Output Ciphertext : {ciphertext_hex}")
print("\n" + "=" * 55)
print("Copy the Key Stream above for decryption.")
print("=" * 55)

         TRIVIUM ENCRYPTION (Part 1)
Key (padded to 80-bit): 646A616D61656A797A61
Generated Nonce/IV:     66E676F91419D9C769F6

Plaintext length : 17 chars → 136 bits
Initialising Trivium (1152 warm-up rounds)...

Output Key Stream : B0EF3ECC0941EE5E6F10D21A0E8B1D6624
Output Ciphertext : F88A52A066029C271F64BD7D7CEA6D0E5D

Copy the Key Stream above for decryption.


## Step 4: Decryption (Part 2)

In [44]:
# ─────────────────── #
# PART 2 — DECRYPTION #
# ─────────────────── #

print("=" * 55)
print("         TRIVIUM DECRYPTION (Part 2)")
print("=" * 55)

#  Inputs 
input_keystream_hex = input("Enter Key Stream (hex): ").strip().upper()
input_ciphertext_hex = input("Enter Ciphertext (hex): ").strip().upper()

#  Recover bits 
# Number of bits = number of hex chars in ciphertext / 2 * 8
num_cipher_bytes = len(input_ciphertext_hex) // 2
num_bits_dec = num_cipher_bytes * 8

ks_bits = hex_keystream_to_bits(input_keystream_hex, num_bits_dec)
ct_bits = hex_keystream_to_bits(input_ciphertext_hex, num_bits_dec)

# OTP Decryption: plaintext = ciphertext XOR keystream 
plain_bits = [c ^ k for c, k in zip(ct_bits, ks_bits)]
recovered_text = bits_to_text(plain_bits)

print(f"\nOutput Plaintext : {recovered_text}")
print("=" * 55)

         TRIVIUM DECRYPTION (Part 2)

Output Plaintext : HelloCryptograph


##  Quick Self-Test with Sample Values from the Lab
This cell will is used to test the helper and trivium cipher methods with absoluteness

> NOTE: to future Drei: Delete this part before submitting dummy

In [45]:
# NOTE: Nonces are random, so keystream WILL differ from the lab sample
#       unless we hard-code the same IV. This test uses the lab's fixed IV.

TEST_PLAINTEXT  = "HelloCryptography"
TEST_KEY_HEX    = "646A616D61656A797A61"   # from lab
TEST_IV_HEX     = "721ED8A325EF88583E2D"   # from lab (fixed for reproducibility)

EXPECTED_KS  = "225CED4D0AB13A68BF782D851EFD13D1"
EXPECTED_CT  = "6A39812165F24811CF0C42E26C9C63B9D4"

print("Running self-test with lab sample values...")
print("-" * 50)

pt_bits  = text_to_bits(TEST_PLAINTEXT)
n_bits   = len(pt_bits)

st = trivium_init(TEST_KEY_HEX, TEST_IV_HEX)
ks = trivium_generate_keystream(st, n_bits)

ks_hex = bits_to_hex(ks)
ct_bits = [p ^ k for p, k in zip(pt_bits, ks)]
ct_hex  = bits_to_hex(ct_bits)

print(f"Plaintext       : {TEST_PLAINTEXT}")
print(f"Key             : {TEST_KEY_HEX}")
print(f"IV              : {TEST_IV_HEX}")
print()
print(f"Key Stream      : {ks_hex}")
print(f"Expected KS     : {EXPECTED_KS}")
print(f"KS Match        : {'YES' if ks_hex == EXPECTED_KS else 'NO'}")
print()
print(f"Ciphertext      : {ct_hex}")
print(f"Expected CT     : {EXPECTED_CT}")
print(f"CT Match        : {'YES' if ct_hex == EXPECTED_CT else 'NO'}")

# --- Decryption check -------------------------------------------------------
rec_bits = [c ^ k for c, k in zip(ct_bits, ks)]
recovered = bits_to_text(rec_bits)
print()
print(f"Decrypted Text  : {recovered}")
print(f"Decrypt Match   : {'YES' if recovered == TEST_PLAINTEXT else 'O'}")

Running self-test with lab sample values...
--------------------------------------------------
Plaintext       : HelloCryptography
Key             : 646A616D61656A797A61
IV              : 721ED8A325EF88583E2D

Key Stream      : 6AF64E3D19E5A022FD819880E3EDEB4AF5
Expected KS     : 225CED4D0AB13A68BF782D851EFD13D1
KS Match        : NO

Ciphertext      : 2293225176A6D25B8DF5F7E7918C9B228C
Expected CT     : 6A39812165F24811CF0C42E26C9C63B9D4
CT Match        : NO

Decrypted Text  : HelloCryptography
Decrypt Match   : YES
